In [ ]:
# We import Libraries 
import pandas as pd 
import numpy as np
import json 

In [ ]:
# We now load the cleaned dataset produced in 01 - data - quality.ipynb
# This ensures the privacy analysis uses the same cleaned data as the data-quality notebook

df_clean = pd.read_csv("cleaned_dataset.csv")

print(f"Total cleaned records: {len(df_clean)}")
df_clean.head()

In [ ]:
# We identify personally identifiable information (PII) columns

pii_columns = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.date_of_birth",
    "applicant_info.zip_code",
]

df_clean[pii_columns].head()

In [ ]:
# EMAIL HASH

import hashlib

def hash_email(email):
    if pd.isna(email):
        return email
    return hashlib.sha256(email.encode()).hexdigest()

df_clean["email_hash"] = df_clean["applicant_info.email"].apply(hash_email)

In [ ]:
# SSN

df_clean["ssn_masked"] = df_clean["applicant_info.ssn"].str.replace(
    r"\d{3}-\d{2}",
    "***-**",
    regex=True
)

In [ ]:
# YEAR 

df_clean["dob_year"] = pd.to_datetime(
    df_clean["applicant_info.date_of_birth"], errors="coerce"
).dt.year

In [ ]:
# BEFORE vs. AFTER

df_clean[[
    "applicant_info.email", "email_hash",
    "applicant_info.ssn", "ssn_masked",
    "applicant_info.date_of_birth", "dob_year"
]].head()

In [ ]:
# We create a privacy-friendly version of the dataset by dropping direct identifiers

df_privacy = df_clean.drop(columns=[
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.date_of_birth",
])
import os
os.makedirs('../data', exist_ok=True) 

df_privacy.to_csv("../data/cleaned_privacy_dataset.csv", index=False)
df_privacy.to_csv("../data/cleaned_privacy_dataset.csv", index=False)

Timestamp missingness undermines model traceability and version control. We recommend automated logging at model inference stage and database-level NOT NULL enforcement